# Module 12: Comparing Agencies Fairly

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Beginner [Topic 14](../Beginner/Topic_14_Comparing_Multiple_Time_Series.md) said
to compare an agency against others over the same months.
[Module 4](Module_04_Why_Small_Agencies_Look_Volatile.md) said to use a funnel
rather than a ranking. Both left one question open: **compared against whom?**

Comparing a 902 officer city force against an eight officer rural department
answers nothing. This module builds peer groups from what agencies actually
look like, which is the problem the WADEPS comparable agencies framework exists
to solve.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")

profile = pd.read_csv(BASE + "agency_profile.csv")
profile[["agency_id", "agency_name", "agency_type", "sworn_officers",
         "population_served", "region"]]

## 2. The problem with matching on size

Officer count alone puts Summit County Sheriff's Office next to agencies it has
nothing operationally in common with. Size is one dimension of several.

In [ ]:
p = profile.copy()
p["short"] = (p["agency_name"].str.replace(" Police Department", "", regex=False)
                              .str.replace(" Sheriff's Office", "", regex=False)
                              .str.replace(" Police", "", regex=False))
by_size = p.reindex(p["sworn_officers"].sort_values().index)
by_size[["short", "agency_type", "sworn_officers", "population_served",
         "violent_crime_rate_per_1000"]].to_string(index=False)

## 3. Gower distance

The agency characteristics are a mix of types: counts, rates, percentages, and
labels like agency type and region. Ordinary distance measures cannot handle
that mixture.

**Gower distance** handles it by treating each variable separately and then
averaging.

- For a **number**: the absolute difference, divided by that variable's range,
  so every variable contributes on a zero to one scale.
- For a **label**: zero if they match, one if they do not.

The result is between 0, meaning identical, and 1, meaning different in every
respect. This is the measure the WADEPS peer group work uses.

In [ ]:
NUMERIC = ["sworn_officers", "population_served", "violent_crime_rate_per_1000",
           "property_crime_rate_per_1000", "budget_share_public_safety_pct",
           "county_population"]
CATEGORICAL = ["agency_type", "region"]


def gower(df, numeric=NUMERIC, categorical=CATEGORICAL, log_scale=()):
    """Distance between every pair of rows, mixing numbers and labels."""
    d = df.copy()
    for c in log_scale:
        d[c] = np.log(d[c])

    n = len(d)
    total = np.zeros((n, n))
    for c in numeric:
        v = d[c].astype(float).values
        total += np.abs(v[:, None] - v[None, :]) / (v.max() - v.min())
    for c in categorical:
        v = d[c].values
        total += (v[:, None] != v[None, :]).astype(float)
    return total / (len(numeric) + len(categorical))

One judgment call is built into the call below. Agency size spans two orders of
magnitude, from 8 officers to 902, so on a raw scale every small agency looks
equally close to every other small agency and Grandview sits alone at the far
end. Taking logs first makes the distance reflect **proportional** difference,
which is the sensible reading of size.

In [ ]:
D = gower(p, log_scale=["sworn_officers", "population_served", "county_population"])
distance = pd.DataFrame(D, index=p["short"], columns=p["short"])

distance.round(2).iloc[:6, :6]

## 4. Who is whose peer

In [ ]:
for a in distance.index:
    near = distance.loc[a].drop(a).nsmallest(3)
    print(f"{a:28s} " + ",  ".join(f"{k} ({v:.2f})" for k, v in near.items()))

The groupings are sensible. Millgate and Northgate pair at 0.09, both small
municipal departments in the east. Summit County and Lakeshore County pair at
0.12, both sheriff's offices in the west.

Two agencies have no close peer at all, and that is a finding rather than a
nuisance.

In [ ]:
loneliness = pd.Series({a: distance.loc[a].drop(a).nsmallest(3).mean()
                        for a in distance.index}).sort_values(ascending=False)
print("mean distance to the three nearest agencies")
print(loneliness.round(2).to_string())

**Grandview** is the largest agency in the state and its nearest peers average
0.31 away, roughly three times the distance between Millgate and Northgate. Any
peer comparison for Grandview is a comparison against agencies that are not
really like it.

**Pinecrest State University** is the only campus force, so its nearest peers
differ from it on agency type by construction.

For both, the right answer is to say the peer group is weak and to compare
against the statewide figure instead, saying so.

## 5. Does the comparison group change the verdict?

In [ ]:
y = (monthly[monthly["year_month"].str[:4] == "2023"]
     .groupby("agency_id", as_index=False)
     .agg(uof=("n_uof", "sum"), arr=("n_arrests", "sum")))
y = y.merge(p[["agency_id", "short"]], on="agency_id").set_index("short")
y["rate"] = 100 * y["uof"] / y["arr"]

state = 100 * y["uof"].sum() / y["arr"].sum()

rows = []
for a in y.index:
    peers = distance.loc[a].drop(a).nsmallest(3).index
    pm = y.loc[peers]
    peer_rate = 100 * pm["uof"].sum() / pm["arr"].sum()
    rows.append((a, y.loc[a, "rate"], y.loc[a, "rate"] - state,
                 peer_rate, y.loc[a, "rate"] - peer_rate))

t = pd.DataFrame(rows, columns=["agency", "rate", "vs state",
                                "peer rate", "vs peers"]).set_index("agency")
t["rank vs state"] = t["vs state"].rank(ascending=False).astype(int)
t["rank vs peers"] = t["vs peers"].rank(ascending=False).astype(int)
t["places moved"] = (t["rank vs state"] - t["rank vs peers"]).abs()
t.round(2).sort_values("places moved", ascending=False)

Most agencies move a place or two. **Pinecrest moves three**, from fifth against
the state to second against its peers, and it is also one of the two agencies
whose peer group is weakest. Both numbers deserve a caveat and neither should be
published alone.

**Harbor Point changes sign.** It is above the statewide rate and below its
peers, because its peers are Cedar Falls and Riverbend, two of the highest rate
agencies in the state. Whether Harbor Point looks good or bad is entirely a
question of who it is placed next to.

## 6. Combine with the funnel

Peer groups say **who** to compare against.
[Module 4](Module_04_Why_Small_Agencies_Look_Volatile.md)'s funnel says
**whether the difference is larger than sampling can explain**. Use both.

In [ ]:
def peer_funnel(agency, y, distance, k=3, z=1.96):
    peers = distance.loc[agency].drop(agency).nsmallest(k)
    pm = y.loc[peers.index]
    pooled = 100 * pm["uof"].sum() / pm["arr"].sum()
    prop = pooled / 100
    se = 100 * np.sqrt(prop * (1 - prop) / y.loc[agency, "arr"])
    diff = y.loc[agency, "rate"] - pooled
    return {"agency": agency, "rate": round(y.loc[agency, "rate"], 2),
            "peer rate": round(pooled, 2), "difference": round(diff, 2),
            "detectable": bool(abs(diff) > z * se),
            "peer distance": round(peers.mean(), 2)}


pd.DataFrame([peer_funnel(a, y, distance) for a in y.index]).set_index("agency")

Read the last two columns together. A difference marked detectable but sitting
next to a large peer distance is a difference from agencies that were never
comparable in the first place.

## 7. What to carry away

| Habit | Why |
|---|---|
| Build peer groups from several characteristics | size alone matches agencies with nothing in common |
| Use Gower distance for mixed data | agency data is always numbers plus labels |
| Take logs of size variables | difference in size is proportional, not absolute |
| Report how far away the peers are | a weak peer group makes the comparison meaningless |
| Report the statewide and the peer comparison | when they disagree, that is the finding |
| Combine the peer group with a funnel | who to compare against, and whether the gap is detectable |
| Let agencies see their peer group first | a comparison nobody accepts changes nothing |

## Exercise

Drop `agency_type` from the distance and rebuild the peer groups. Who moves,
and what does that tell you about what the variable was doing?

In [ ]:
# Fill in the blank, then run.
CATEGORICAL_TO_USE = None       # try ["region"]

if CATEGORICAL_TO_USE is not None:
    D2 = gower(p, categorical=CATEGORICAL_TO_USE,
               log_scale=["sworn_officers", "population_served", "county_population"])
    d2 = pd.DataFrame(D2, index=p["short"], columns=p["short"])
    for a in ["Pinecrest State University", "Two Rivers Tribal", "Summit County"]:
        before = list(distance.loc[a].drop(a).nsmallest(3).index)
        after = list(d2.loc[a].drop(a).nsmallest(3).index)
        print(f"{a}\n   with agency type:    {before}\n   without agency type: {after}\n")
else:
    print("Set CATEGORICAL_TO_USE above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
CATEGORICAL_TO_USE = ["region"]
```

Pinecrest and Two Rivers Tribal change peers, and the sheriff's offices change
less. Without `agency_type`, a campus force and a tribal department become
matchable with any municipal department of similar size, because nothing else
in the variable list captures the difference in what those agencies are for.

The lesson is about variable choice rather than about method. Gower distance
will faithfully average whatever you hand it, and a characteristic left out of
the list is a characteristic the peer groups will ignore. Deciding what belongs
in the list is a domain judgment, which is why the WADEPS framework validates
its peer groups with people who know the agencies rather than only with a
distance measure.

</details>

---

**Next:** Part IV of the Intermediate series, on forecasting and change
detection: the baseline forecasts you have to beat, exponential smoothing,
measuring forecast error, and telling whether something changed.

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*